# Focus

This notebooks focus is on checking the data and cleaning data from yFinance

# Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Make sure data and processed directories exist

In [2]:
RAW_DATA_PATH = Path("../data/raw")
PROCESSED_DATA_PATH = Path("../data/processed")

In [3]:

if not RAW_DATA_PATH.exists():
    print(f"Creating directory: {RAW_DATA_PATH}")
    RAW_DATA_PATH.mkdir(parents=True, exist_ok=True)
else:
    print(f"Directory already exists: {RAW_DATA_PATH}")
if not PROCESSED_DATA_PATH.exists():
    print(f"Creating directory: {PROCESSED_DATA_PATH}")
    PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)
else:
    print(f"Directory already exists: {PROCESSED_DATA_PATH}")

Directory already exists: ..\data\raw
Directory already exists: ..\data\processed


# Setup Adjusted Close Data

First, I loaded the file normally to inspect its structure. 

Then I reloaded it with the date column as the index so the dataset is easier to use for time-series analysis.

In [4]:
adj_close_0 = pd.read_csv(
    RAW_DATA_PATH / "adjusted_close_prices.csv",
)

adj_close_0.head()

,Date,AAPL,AGG,AMZN,GLD,GOOGL,JNJ,JPM,MSFT,NVDA,PG,QQQ,SPY,TSLA,XOM
0,2018-01-02,40.267078,85.561790,59.450500,125.150002,53.188869,110.130806,85.901253,78.699890,4.922531,72.280716,150.057220,235.954315,21.368668,58.185493
1,2018-01-03,40.260063,85.569595,60.209999,124.820000,54.096317,111.182846,85.988792,79.066162,5.246500,72.192993,151.515320,237.446686,21.150000,59.328262
2,2018-01-04,40.447067,85.514763,60.479500,125.459999,54.306446,111.174911,87.220634,79.762085,5.274156,72.703316,151.780396,238.447510,20.974667,59.410374
3,2018-01-05,40.907578,85.459900,61.457001,125.330002,55.026569,112.092522,86.660721,80.750931,5.318850,72.751152,153.304764,240.036621,21.105333,59.362476
4,2018-01-08,40.755634,85.436424,62.343498,125.309998,55.220840,112.234863,86.788727,80.833359,5.481825,73.133888,153.901215,240.475464,22.427334,59.629349


In [5]:
# Make index a datetime index
adj_close = pd.read_csv(
    RAW_DATA_PATH / "adjusted_close_prices.csv",
    index_col=0,
    parse_dates=True
)

adj_close.head()

,AAPL,AGG,AMZN,GLD,GOOGL,JNJ,JPM,MSFT,NVDA,PG,QQQ,SPY,TSLA,XOM
Date,,,,,,,,,,,,,,
2018-01-02,40.267078,85.561790,59.450500,125.150002,53.188869,110.130806,85.901253,78.699890,4.922531,72.280716,150.057220,235.954315,21.368668,58.185493
2018-01-03,40.260063,85.569595,60.209999,124.820000,54.096317,111.182846,85.988792,79.066162,5.246500,72.192993,151.515320,237.446686,21.150000,59.328262
2018-01-04,40.447067,85.514763,60.479500,125.459999,54.306446,111.174911,87.220634,79.762085,5.274156,72.703316,151.780396,238.447510,20.974667,59.410374
2018-01-05,40.907578,85.459900,61.457001,125.330002,55.026569,112.092522,86.660721,80.750931,5.318850,72.751152,153.304764,240.036621,21.105333,59.362476
2018-01-08,40.755634,85.436424,62.343498,125.309998,55.220840,112.234863,86.788727,80.833359,5.481825,73.133888,153.901215,240.475464,22.427334,59.629349


In [6]:
adj_close.shape

(2010, 14)

In [7]:
adj_close.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 2010 entries, 2018-01-02 to 2025-12-30
Data columns (total 14 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AAPL    2010 non-null   float64
 1   AGG     2010 non-null   float64
 2   AMZN    2010 non-null   float64
 3   GLD     2010 non-null   float64
 4   GOOGL   2010 non-null   float64
 5   JNJ     2010 non-null   float64
 6   JPM     2010 non-null   float64
 7   MSFT    2010 non-null   float64
 8   NVDA    2010 non-null   float64
 9   PG      2010 non-null   float64
 10  QQQ     2010 non-null   float64
 11  SPY     2010 non-null   float64
 12  TSLA    2010 non-null   float64
 13  XOM     2010 non-null   float64
dtypes: float64(14)
memory usage: 235.5 KB


In [8]:
# Check for missing vals
missing_counts = adj_close.isna().sum()
missing_percent = adj_close.isna().mean() * 100

missing_summary = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_percent": missing_percent
}).sort_values("missing_percent", ascending=False)

missing_summary

,missing_count,missing_percent
AAPL,0,0.0
AGG,0,0.0
AMZN,0,0.0
GLD,0,0.0
GOOGL,0,0.0
JNJ,0,0.0
JPM,0,0.0
MSFT,0,0.0
NVDA,0,0.0
PG,0,0.0


This table shows how many missing adjusted close prices each asset has which shows 0 missing values. Missing values need to be checked before calculating final returns because missing prices can create incorrect return values for model

In [9]:
# In case future missing data 
adj_close_clean = adj_close.dropna()

adj_close_clean.shape

(2010, 14)

In [10]:
print("Original rows:", adj_close.shape[0])
print("Clean rows:", adj_close_clean.shape[0])
print("Rows removed:", adj_close.shape[0] - adj_close_clean.shape[0])

Original rows: 2010
Clean rows: 2010
Rows removed: 0


For this first preprocessing version, rows with missing adjusted close prices are removed. This keeps all assets aligned on the same trading dates, 

This is important for calculating returns, correlations, and portfolio metrics.

In [11]:
daily_returns_clean = adj_close_clean.pct_change().dropna()

daily_returns_clean.head()

,AAPL,AGG,AMZN,GLD,GOOGL,JNJ,JPM,MSFT,NVDA,PG,QQQ,SPY,TSLA,XOM
Date,,,,,,,,,,,,,,
2018-01-03,-0.000174,0.000091,0.012775,-0.002637,0.017061,0.009553,0.001019,0.004654,0.065814,-0.001214,0.009717,0.006325,-0.010233,0.019640
2018-01-04,0.004645,-0.000641,0.004476,0.005127,0.003884,-0.000071,0.014326,0.008802,0.005271,0.007069,0.001749,0.004215,-0.008290,0.001384
2018-01-05,0.011386,-0.000642,0.016163,-0.001036,0.013260,0.008254,-0.006420,0.012397,0.008474,0.000658,0.010043,0.006664,0.006230,-0.000806
2018-01-08,-0.003714,-0.000275,0.014425,-0.000160,0.003530,0.001270,0.001477,0.001021,0.030641,0.005261,0.003891,0.001828,0.062638,0.004496
2018-01-09,-0.000115,-0.002753,0.004676,-0.004628,-0.001274,0.015857,0.005069,-0.000680,-0.000271,-0.007305,0.000062,0.002264,-0.008085,-0.004246


In [12]:
daily_returns_clean.shape

(2009, 14)

In [13]:
daily_returns_clean.describe()

,AAPL,AGG,AMZN,GLD,GOOGL,JNJ,JPM,MSFT,NVDA,PG,QQQ,SPY,TSLA,XOM
count,2009.000000,2009.000000,2009.000000,2009.000000,2009.000000,2009.000000,2009.000000,2009.000000,2009.000000,2009.000000,2009.000000,2009.000000,2009.000000,2009.000000
mean,0.001140,0.000077,0.000914,0.000623,0.001074,0.000385,0.000822,0.001065,0.002334,0.000415,0.000820,0.000605,0.002323,0.000538
std,0.019401,0.003669,0.021680,0.009535,0.019523,0.012333,0.018298,0.017861,0.032318,0.012591,0.015153,0.012260,0.040118,0.018996
min,-0.128647,-0.040011,-0.140494,-0.064269,-0.116342,-0.100378,-0.149649,-0.147390,-0.187559,-0.087373,-0.119788,-0.109423,-0.210628,-0.122248
25%,-0.007925,-0.001631,-0.010250,-0.004385,-0.008510,-0.005351,-0.007638,-0.007372,-0.015115,-0.005599,-0.005957,-0.004280,-0.019163,-0.009448
50%,0.001187,0.000178,0.001084,0.000646,0.001419,0.000530,0.000872,0.001254,0.002814,0.000730,0.001365,0.000899,0.001535,0.000481
75%,0.011071,0.001849,0.012258,0.005667,0.011264,0.006217,0.009361,0.010266,0.019810,0.006882,0.008557,0.006519,0.021914,0.010647
max,0.153289,0.023721,0.135359,0.048530,0.102244,0.079977,0.180125,0.142169,0.243697,0.120091,0.120031,0.105019,0.226900,0.126868


## Check for weird infinite or weird vals

In [14]:
np.isinf(daily_returns_clean).sum()

AAPL     0
AGG      0
AMZN     0
GLD      0
GOOGL    0
JNJ      0
JPM      0
MSFT     0
NVDA     0
PG       0
QQQ      0
SPY      0
TSLA     0
XOM      0
dtype: int64

In [15]:
daily_returns_clean.isna().sum()

AAPL     0
AGG      0
AMZN     0
GLD      0
GOOGL    0
JNJ      0
JPM      0
MSFT     0
NVDA     0
PG       0
QQQ      0
SPY      0
TSLA     0
XOM      0
dtype: int64

# Save Processed Cleaned Data

In [16]:
adj_close_clean.to_csv(PROCESSED_DATA_PATH / "adjusted_close_clean.csv")
daily_returns_clean.to_csv(PROCESSED_DATA_PATH / "daily_returns_clean.csv")

# High Level Processed Data Summary

In [17]:
TRADING_DAYS = 252

In [18]:
processed_summary = pd.DataFrame({
    "mean_daily_return": daily_returns_clean.mean(),
    "daily_volatility": daily_returns_clean.std(),
    "annualized_return": daily_returns_clean.mean() * TRADING_DAYS,
    "annualized_volatility": daily_returns_clean.std() * np.sqrt(TRADING_DAYS),
    "min_daily_return": daily_returns_clean.min(),
    "max_daily_return": daily_returns_clean.max()
})

In [19]:
processed_summary

,mean_daily_return,daily_volatility,annualized_return,annualized_volatility,min_daily_return,max_daily_return
AAPL,0.001140,0.019401,0.287326,0.307982,-0.128647,0.153289
AGG,0.000077,0.003669,0.019346,0.058242,-0.040011,0.023721
AMZN,0.000914,0.021680,0.230284,0.344154,-0.140494,0.135359
GLD,0.000623,0.009535,0.156912,0.151369,-0.064269,0.048530
GOOGL,0.001074,0.019523,0.270602,0.309912,-0.116342,0.102244
JNJ,0.000385,0.012333,0.096933,0.195777,-0.100378,0.079977
JPM,0.000822,0.018298,0.207217,0.290465,-0.149649,0.180125
MSFT,0.001065,0.017861,0.268446,0.283531,-0.147390,0.142169
NVDA,0.002334,0.032318,0.588212,0.513032,-0.187559,0.243697
PG,0.000415,0.012591,0.104637,0.199871,-0.087373,0.120091


## Save to CSV

In [20]:
processed_summary.to_csv(PROCESSED_DATA_PATH / "processed_summary_stats.csv")

# Final Summary

In this notebook it was focused on loading in raw adjusted close price data, checking missing vals, align the assets by date, calc clean daily returns, and save processed datasets.

### Files Created
- `data/processed/adjusted_close_clean.csv`
- `data/processed/daily_returns_clean.csv`
- `data/processed/processed_summary_stats.csv` 